## Algorithm Convergence & Warning Management

* **Iterative Solvers:** 
Many machine learning algorithms (like Logistic Regression or Neural Networks) do not find the perfect mathematical answer instantly. 

They are "iterative," meaning they take a guess, calculate the error, and adjust their weights step-by-step.

"Convergence" occurs when the algorithm has found the optimal solution and further adjustments no longer improve the model. 

If you restrict the maximum iterations (e.g., `max_iter=10`), the solver may be forced to stop before it reaches convergence.

* **Handling Warnings:** 
When an algorithm fails to converge within the allowed iterations, libraries like Scikit-Learn will throw a `ConvergenceWarning`. 

In Python, you can use the `warnings` library (`warnings.filterwarnings("ignore")`) to suppress these messages during intentional experimentation, keeping your output console clean.

---

## Reproducibility and Randomness

* **Random Seeding:** 
Algorithms rely heavily on random number generation for tasks like shuffling data, splitting datasets, and initializing starting weights. 

By setting a specific "seed" (e.g., `random_state=42`), you lock the random number generator. 

This ensures that your code behaves exactly the same way every time you run it, making your experiments reproducible.

* **Performance Variance:** 
If you do *not* lock the random seed, your `train_test_split` will grab different data points every time. 

Because the model is learning from a slightly different training set, its final accuracy score will naturally fluctuate. 

Evaluating a model across multiple random seeds is a great way to see how stable (or volatile) its performance is on different data distributions.

---

## Hyperparameter Sensitivity & Model Comparison

* **Iterative Tuning:** 
This is the process of systematically testing different hyperparameter values to find the best configuration. 

For example, you might loop through a list of iteration limits (`[1, 10, 50, 500, 5000]`) to observe the direct trade-off between how long a model takes to train and how accurate its predictions become.

* **Under vs Trained Models:** 
Comparing a heavily restricted model (e.g., an under-trained model with `max_iter=10`) directly against a fully trained model (e.g., `max_iter=5000`) helps you establish a performance baseline. 

It proves experimentally whether the extra training time actually yields a significantly better model.

* **Disagreement Analysis:** 
Overall accuracy scores can hide nuanced model behaviors. 

Disagreement analysis involves using array logic (e.g., `np.where(y_pred_small != y_pred_big)`) to isolate the exact data points where two models gave different answers. 

Inspecting these specific edge cases helps you understand *how* the models differ in their logic, not just *how much* they differ in accuracy.

---

## Deep Learning Fundamentals (PyTorch)

* **Tensors:** 
Tensors are the fundamental data structure in PyTorch. 

They look and act almost exactly like NumPy arrays (matrices of numbers), but they have special properties that allow them to be processed massively in parallel on GPUs, and they can automatically track their own mathematical gradients for machine learning.

* **Neural Networks:** 
In PyTorch, you build networks using the `torch.nn` module. 

A simple Logistic Regression model can be entirely recreated in deep learning as a single dense layer (`nn.Linear`). 

For example, `nn.Linear(64, 10)` creates a layer that takes in 64 pixel features and outputs 10 probability logits (one for each digit).

* **The Training Loop:** 
Unlike Scikit-Learn where you just call `model.fit()`, PyTorch requires you to explicitly write the steps of the iterative solver. 

A standard deep learning training loop consists of four mandatory steps:
-  **Forward Pass:** Pass the training data through the model to get raw predictions (`logits = model(X)`).

- **Calculate Loss:** Measure how far off the predictions are from the true labels using a loss function (`loss = loss_fn(logits, y)`).

- **Backward Pass (Backpropagation):** First, clear out old gradients (`optimizer.zero_grad()`). Then, calculate the new gradients by tracing the math backward (`loss.backward()`).

- **Optimize (Step):** Adjust the model's weights in the right direction to reduce the loss (`optimizer.step()`).

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning

# PyTorch Imports
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# SETUP: Load the Data
# ==========================================
digits = load_digits()
X = digits.data
y = digits.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ==========================================
# TOPIC 1: ALGORITHM CONVERGENCE & WARNINGS
# ==========================================
print("--- 1. Algorithm Convergence & Warning Management ---")
# We intentionally set max_iter too low so the model cannot converge.
# We use the warnings library to suppress the messy red text in our console.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

print("Training Logistic Regression with max_iter=10 (ConvergenceWarning suppressed)...")
model_under = LogisticRegression(max_iter=10, random_state=42)
model_under.fit(X_train, y_train)
print("Training complete without console spam!\n")


# ==========================================
# TOPIC 2: REPRODUCIBILITY & VARIANCE
# ==========================================
print("--- 2. Reproducibility & Performance Variance ---")
# If we don't lock the random seed, our data splits differently every time, 
# causing natural fluctuations in our model's final accuracy.
seeds = [10, 42, 99]
print("Observing Accuracy Variance across different random data splits:")

for seed in seeds:
    Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    temp_model = LogisticRegression(max_iter=1000)
    temp_model.fit(Xt, yt)
    acc = accuracy_score(yv, temp_model.predict(Xv))
    print(f"  Random Seed {seed}: Accuracy = {acc:.3f}")
print()


# ==========================================
# TOPIC 3: MODEL COMPARISON & DISAGREEMENT
# ==========================================
print("--- 3. Hyperparameter Tuning & Disagreement Analysis ---")
# We test a severely under-trained model against a fully trained one.
model_small = LogisticRegression(max_iter=10, random_state=42)
model_small.fit(X_train, y_train)
y_pred_small = model_small.predict(X_test)

model_big = LogisticRegression(max_iter=5000, random_state=42)
model_big.fit(X_train, y_train)
y_pred_big = model_big.predict(X_test)

print(f"Under-trained Model (max_iter=10) Accuracy:   {accuracy_score(y_test, y_pred_small):.3f}")
print(f"Fully-trained Model (max_iter=5000) Accuracy: {accuracy_score(y_test, y_pred_big):.3f}")

# Array logic to find the exact data points where the two models gave different answers
disagree_idx = np.where(y_pred_small != y_pred_big)[0]
print(f"The two models disagreed on exactly {len(disagree_idx)} test samples.")
print(f"The first 5 test indices they disagreed on are: {disagree_idx[:5]}\n")


# ==========================================
# TOPIC 4: PYTORCH FUNDAMENTALS
# ==========================================
print("--- 4. PyTorch: Tensors, Neural Networks, & Training Loop ---")

# Step A: Convert NumPy arrays into PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# Step B: Define the Neural Network Architecture
# A single Linear layer (64 input pixels -> 10 output digit probabilities)
torch_model = nn.Linear(64, 10)

# Step C: Setup Loss Function and Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(torch_model.parameters(), lr=0.01)

# Step D: The Mandatory 4-Step PyTorch Training Loop
print("Starting explicit PyTorch Training Loop...")
for epoch in range(101): # Loop over the data 100 times
    
    # 1. Forward Pass: Generate raw predictions (logits)
    logits = torch_model(X_train_t)
    
    # 2. Calculate Loss: Measure the error
    loss = loss_fn(logits, y_train_t)
    
    # 3. Backpropagation: Clear old gradients, calculate new ones
    optimizer.zero_grad()
    loss.backward()
    
    # 4. Optimize (Step): Adjust the network's weights
    optimizer.step()
    
    # Print an update every 25 epochs
    if epoch % 25 == 0:
        with torch.no_grad(): # Turn off gradient tracking for evaluation
            preds = torch_model(X_test_t).argmax(dim=1)
            acc = (preds == y_test_t).float().mean().item()
        print(f"  Epoch {epoch:>3}: Loss = {loss.item():.3f} | Test Accuracy = {acc:.3f}")